<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/E09/mva_2026_E09_03_ward.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_E09_03_ward.ipynb Ward法による階層的クラスタリング

- 学籍番号 [ ]
- 氏名 [ ]

In [17]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline

# テストデータの生成
from sklearn.mixture import GaussianMixture

# 階層的クラスタリング
from sklearn.cluster import AgglomerativeClustering

# デンドログラム
from scipy.cluster.hierarchy import linkage, dendrogram

# クラスタリングの評価
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

%pip install -q japanize_matplotlib
# -q セルへの出力を抑制
import japanize_matplotlib # 日本語フォント

##ガウス混合分布データの生成(K=3)

In [ ]:
# ガウス混合分布から疑似データを生成
np.random.seed(42)
K0=3
n_samples = 300

# 3つの構成要素のパラメータ設定
means = np.array([[-1, -1], [1, 3], [3, -1]])
covs = np.array([[[2.0, 0.5], [0.5, 2.0]],
                 [[1.5, -0.3], [-0.3, 2.0]],
                 [[0.8, 0.0], [0.0, 0.8]]])
weights = np.array([0.4, 0.3, 0.3])

# sklearnのGaussianMixtureインスタンスを作成し、パラメータを設定
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm.fit(np.random.randn(10, 2))  # パラメータを手動設定するために一度ダミーで初期化

# 事前に定義した平均、共分散、重みを設定
gmm.weights_ = weights
gmm.means_ = means
gmm.covariances_ = covs
gmm.precisions_cholesky_ = np.linalg.cholesky(np.linalg.inv(covs))

# ガウス混合モデルからサンプルを生成
X, y = gmm.sample(n_samples)

In [ ]:
# 生成したデータのプロット
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.6, edgecolors='k')
plt.title(f'生成されたガウス混合分布のデータ(K={K0})')
plt.xlabel("x1")
plt.ylabel("x2")
plt.grid(True)
plt.show()

## Ward法による階層的クラスタリングの可視化

### K=3

In [ ]:
# from sklearn.cluster import AgglomerativeClustering

# クラスターの個数の指定
K = 3

# Ward法による階層的クラスタリングモデルの定義
ward = AgglomerativeClustering(n_clusters=K, linkage='ward')
ward.fit(X)
# 教師なし学習なので，別にpredictを行わない
# 訓練データXに対するlabelがつけられているのでそれを取り出す
labels=ward.labels_

# 先頭のデータのみを出力
print("予測ラベル（先頭10個）:", labels[:10])

# 各クラスタの平均値を中心点として計算
centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])

# クラスタリング結果のプロット
plt.figure(figsize=(8, 6))

# クラスター数 K に応じて動的に色を生成
cormap = plt.colormaps.get_cmap('rainbow')
colors = [cormap(i / K) for i in range(K)]

for k in range(K):
    cluster_data = X[labels == k]
    plt.scatter(cluster_data[:, 0], cluster_data[:, 1], color=colors[k], alpha=0.5, label=f'クラスター {k}', edgecolors='k')

# 算出された中心点のプロット
plt.scatter(centroids[:, 0], centroids[:, 1], c='yellow', marker='X', s=200, edgecolors='black', linewidths=2, label='重心（平均）')

plt.title(f'Ward法によるクラスタリング結果(K={K})')
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.grid(True)
plt.show()

### K=2

In [ ]:
# from sklearn.cluster import AgglomerativeClustering

# クラスターの個数の指定
K = 2

# Ward法による階層的クラスタリングモデルの定義と学習
ward = AgglomerativeClustering(n_clusters=K, linkage='ward')
ward.fit(X)
# 教師なし学習なので，別にpredictを行わない
# 訓練データXに対するlabelがつけられているのでそれを取り出す
labels=ward.labels_
# 各クラスタの平均値を中心点として計算
centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])

# クラスタリング結果のプロット
plt.figure(figsize=(8, 6))

# クラスター数 K に応じて動的に色を生成
cormap = plt.colormaps.get_cmap('rainbow')
colors = [cormap(i / K) for i in range(K)]

for k in range(K):
    cluster_data = X[labels == k]
    plt.scatter(cluster_data[:, 0], cluster_data[:, 1], color=colors[k], alpha=0.5, label=f'クラスター {k}', edgecolors='k')

# 算出された中心点のプロット
plt.scatter(centroids[:, 0], centroids[:, 1], c='yellow', marker='X', s=200, edgecolors='black', linewidths=2, label='重心（平均）')

plt.title(f'Ward法によるクラスタリング結果(K={K})')
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.grid(True)
plt.show()

### デンドログラムの描画と画像ファイル保存

K=1になるまで併合を繰り返すために
```
distance_threshold=0, n_clusters=None
```
と指定しています．

モデル model に保存された情報を，scipy の dendrogram関数を利用して描画しています(scikit-learnにいい関数がないので）

また，デンドログラムを見やすくするために，p=5分岐までに制限して描いています

In [ ]:
# scikit-learnのAgglomerativeClusteringからデンドログラムを描画するためのヘルパー関数
def plot_dendrogram(model, **kwargs):
    # 各ノードの下にあるサンプル数のカウントを作成
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # 葉ノード（個別データ）
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack([
        model.children_,
        model.distances_,
        counts
    ]).astype(float)

    # scipyのdendrogramを利用してプロット
    dendrogram(linkage_matrix, **kwargs)

# Ward法（scikit-learn）で全結合木を構築するため、n_clusters=None, distance_threshold=0 に設定
model = AgglomerativeClustering(distance_threshold=0, n_clusters=None, linkage='ward')
model = model.fit(X)

p=5

plt.figure(figsize=(12, 6))
plt.title("デンドログラム (sklearnのAgglomerativeClusteringによるWard法)")
# 樹形図を描画
plot_dendrogram(model, truncate_mode='level', p=p)  # 表示が見やすくなるようレベルpまで制限して表示
plt.xlabel("データ点のインデックス（または子ノードの数）")
plt.ylabel("結合距離 (Ward法)")
plt.grid(True, axis='y')

# 画像ファイルとして保存 (PNG形式、高解像度 300dpi)
output_image_path = "/content/dendrogram_ward.png"
plt.savefig(output_image_path, dpi=300, bbox_inches='tight')

plt.show()

### K=4

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# クラスターの個数の指定
K = 4

# Ward法による階層的クラスタリングモデルの定義と学習
ward = AgglomerativeClustering(n_clusters=K, linkage='ward')
ward.fit(X)
# 教師なし学習なので，別にpredictを行わない
# 訓練データXに対するlabelがつけられているのでそれを取り出す
labels=ward.labels_
labels=ward.labels_

# 各クラスタの平均値を中心点として計算
centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])

# クラスタリング結果のプロット
plt.figure(figsize=(8, 6))

# クラスター数 K に応じて動的に色を生成
cormap = plt.colormaps.get_cmap('rainbow')
colors = [cormap(i / K) for i in range(K)]

for k in range(K):
    cluster_data = X[labels == k]
    plt.scatter(cluster_data[:, 0], cluster_data[:, 1], color=colors[k], alpha=0.5, label=f'クラスター {k}', edgecolors='k')

# 算出された中心点のプロット
plt.scatter(centroids[:, 0], centroids[:, 1], c='yellow', marker='X', s=200, edgecolors='black', linewidths=2, label='重心（平均）')

plt.title(f'Ward法によるクラスタリング結果(K={K})')
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.grid(True)
plt.show()

### クラスタリング結果の評価

In [ ]:
# from sklearn.cluster import AgglomerativeClustering
# from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# 評価用のKの値のリスト
k_values = range(2,7)
eval_results = []

for k_val in k_values:
    # 各KでWard法を実行
    ward_model = AgglomerativeClustering(n_clusters=k_val, linkage='ward')
    # predict(X)の代わりにfit_predict(X)を使用してクラスタラベルを取得します
    cluster_labels = ward_model.fit_predict(X)

    # 各種評価指標の計算 (正解データ y を使わない評価)
    #sil = silhouette_score(X, cluster_labels)
    ch = calinski_harabasz_score(X, cluster_labels)
    #db = davies_bouldin_score(X, cluster_labels)

    eval_results.append({
        'K': k_val,
#        'シルエット係数 (↑高評価)': sil,
        'Calinski-Harabasz (CH) 指数 (↑高評価)': ch
#        ,
#        'Davies-Bouldin 指数 (↓高評価)': db
    })

# 結果をDataFrameにまとめて表示
df_eval = pd.DataFrame(eval_results)
display(df_eval)